In [1]:
import numpy as np
import math
import tqdm
import struct
import matplotlib.pyplot as plt
import csv
import os
from tqdm.auto import tqdm
from binfile import *
from PIL import Image
import pygame

# Create a drifting grating
def find_files(path):
    return sorted([f for f in os.listdir(path) if (os.path.isfile(os.path.join(path, f)) and os.path.splitext(f)[1] in [".vec",".bin"])])       #If no, the path is considered as a folder and return the name of all the files in alphabetic order


root = r"/home/guiglaz/Documents/stim generation/StimulusPlayer/"
BIN = os.path.join(root,"BIN/")
VEC = os.path.join(root,"VEC/")

pygame 2.6.1 (SDL 2.28.4, Python 3.9.16)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
#Select Vec file
vec_files = find_files(VEC)
print(*['{} : {}'.format(i,vec_file) for i, vec_file in enumerate(vec_files)], sep="\n")
vec_number = int(input("\nSelect VEC : "))
vec_file = vec_files[vec_number]
print(f"\nSelected recording : {vec_file} \n")

vec_trigs = np.loadtxt(os.path.join(VEC,vec_file))[1:]

print("-------------------------------\n")

#Select Bin File
bin_files = find_files(BIN)
print(*['{} : {}'.format(i,bin_file) for i, bin_file in enumerate(bin_files)], sep="\n")
bin_number = int(input("\nSelect BIN : "))
bin_file = bin_files[bin_number]
print(f"\nSelected recording : {bin_file} \n")

print("-------------------------------")

MEA = int(input("\nEnter MEA number 2 or 3 :"))
if MEA==2: 
    threshold  = 150e+3
    pxl_size_dmd = 3.5
    size_dmd = [864, 864]      # dimensions of the DMD, in pixels
    polarity = 0
if MEA==3:
    threshold  = 170e+3   
    size_dmd = [760, 1020]      # dimensions of the DMD, in pixels
    pxl_size_dmd = 2.5          # The size of one pixel of the DMD in µm? on the camera or in reality?
    polarity = -1
    
binObj = BinFile(os.path.join(BIN,bin_file), size_dmd[0], size_dmd[1], MEA, mode='r')

print("\n-------------------------------")


FPS = int(input("\nFrame rate : "))


# Color dictionary for the second dot column
color_dict = {
    0: (255, 255, 255),  # white
    1: (255, 0, 0),      # red
    2: (0, 255, 0),      # green
    3: (128, 0, 128),    # violet
    4: (50, 50, 50)      # dark
}

0 : moving_bar_texture_dh_V8_29dhspots.vec

Selected recording : moving_bar_texture_dh_V8_29dhspots.vec 

-------------------------------

0 : moving_bar_texture_dh_V8_rig_3.bin

Selected recording : moving_bar_texture_dh_V8_rig_3.bin 

-------------------------------

-------------------------------


# Plotting

In [6]:
# Initialize Pygame
pygame.init()

# Constants
WIDTH, HEIGHT = 50+size_dmd[1]+50+250, size_dmd[0]+100
DOT_RADIUS = 50
TRIANGLE_SIZE = 50
FONT_SIZE = 50

# Initialize Pygame window
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Information Display")

def load_image(idx,binObj, polarity):
    try :
        image = binObj.read_frame(idx)  # Assuming this returns a normalized array between 0 and 1
        image = abs(np.array(255*(image+polarity)).astype(np.uint16))
    except AssertionError:
        print("Index out of range, displaying first frame !")
        idx = 0
        image = binObj.read_frame(idx)
        image = abs(np.array(255*(image+polarity)).astype(np.uint16))
        
#    surface = pygame.surfarray.make_surface(image)
    surface = pygame.surfarray.make_surface(np.stack((image,)*3, axis=-1))  # Convert to 3-channel greyscale
    return surface


# Function to draw a triangle
def draw_triangle(surface, color, position):
    points = [
        (position[0], position[1] + TRIANGLE_SIZE),
        (position[0] - TRIANGLE_SIZE, position[1] - TRIANGLE_SIZE),
        (position[0] + TRIANGLE_SIZE, position[1] - TRIANGLE_SIZE),
    ]
    pygame.draw.polygon(surface, color, points)

# Assuming vec_trigs is defined somewhere and contains the data
data = vec_trigs[10000:11000]  
current_index = 0  # Change this to display a different row

# Set up the clock
clock = pygame.time.Clock()
paused = False  # Track the paused state

# Main loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:  # Spacebar to pause/resume
                paused = not paused
        
    # Clear the screen
    screen.fill((128, 128, 128))
    if not paused:
        # Draw the current row of data
        row = data[current_index]
        y_offset = 50  # Fixed position since displaying one row
        x_offset = 50+size_dmd[1]+50
        # Draw the image on the left side
        image = load_image(int(row[1]),binObj, polarity)
        image = pygame.transform.scale(image, (size_dmd[1], size_dmd[0]))  # Scale the image as needed
        screen.blit(image, (50, y_offset))  # Position the image

        # Prepare the text information on the right side
        font = pygame.font.Font(None, FONT_SIZE)
        # Fifth column: text (bottom)
        text_surface_5 = font.render(str(int(row[4]))[:-2], True, (255, 255, 255))
        text_width = text_surface_5.get_width()
        screen.blit(text_surface_5, (x_offset+100 - text_width // 2, y_offset + 50))

        # Fourth column: triangle (above text)
        triangle_color = (255, 0, 0) if row[3] == 1 else (50, 50, 50)
        draw_triangle(screen, triangle_color, (x_offset+100, y_offset + 200))

        # First column: colored dot (above triangle)
        dot_color = (0, 255, 0) if row[0] == 1 else (50, 50, 50)
        pygame.draw.circle(screen, dot_color, (x_offset+100, y_offset + 400), DOT_RADIUS)

        # Third column: colored dot (above first dot)
        dot_color_2 = color_dict.get(row[2], (255, 255, 255))
        pygame.draw.circle(screen, dot_color_2, (x_offset+100, y_offset + 600), DOT_RADIUS)

        # Prepare the text information on the right side
        font = pygame.font.Font(None, FONT_SIZE//2)
        # Fifth column: text (bottom)
        text_surface_1 = font.render(str(int(row[1])), True, (255, 255, 255))
        text_width = text_surface_1.get_width()
        screen.blit(text_surface_1, (x_offset-45, y_offset + size_dmd[0]- 15))

        # Update the display
        pygame.display.flip()

        # Move to the next index
        current_index = (current_index + 1) % len(data)

        # Control the speed of the cycle
        clock.tick(FPS)

# Quit Pygame
pygame.quit()


Index out of range, displaying first frame !
